# Evaluation on Blotto: empirical exploitability (Table 2, BLT column)
Every frozen defender plays the same 500 pre-drawn episodes (responsiveness, valuations, reachability, and initial state) against four admissible attackers:

1. `Trained P2`, the trained physical attacker,
2. `Critic-BR P2`, a myopic best response to the defender's current mixed action under the critic,
3. `Equilibrium P2`, the equilibrium column of the critic's stage game, and
4. `BR-PPO P2`, a recurrent-PPO best response trained from scratch against that defender (`best_response.ipynb`).

Every attacker is restricted to its admissible allocations. Returns are the defender's discounted payoff (γ = 0.95, 50 rounds), so attackers minimize them. A defender's worst-case return $\widehat W$ is its lowest mean over the attackers, and its empirical exploitability is $\widehat\Delta = \widehat V^\star - \widehat W$, where $\widehat V^\star$ is the best worst-case return among the evaluated defenders. Every defender samples its action from its own distribution, as at deployment.

In [ ]:
%run Initialize.ipynb

## Settings, trained models, and the shared episodes

In [ ]:
GAMMA = 0.95
num_episodes = 500
SAVE_DIR = "compared"

Context_Model = Load(TSTransformer().to(device), "encoder.pth")
Z_S_ENC       = Load(StateGRU().to(device), "z_s_enc.pth")
critic        = Load(Critic().to(device), "critic.pth")
Player1       = Load(Player1Actor().to(device), "player1.pth")
BASELINES     = Load_Baselines(SAVE_DIR)
Player2       = Load(Player2Actor().to(device), "player2.pth")

episode_configs = []
cfg_rng = np.random.RandomState(12345)
for _ in range(num_episodes):
    Theta = np.zeros(Len_Theta)
    Theta[0:N] = cfg_rng.choice(RHO_VALUES, size=N)
    Reachable = np.zeros(N)
    Reachable[cfg_rng.choice(N, size=3, replace=False)] = 1.0
    valued = cfg_rng.choice(N, size=3, replace=False)
    Weights = np.zeros(N)
    Weights[valued] = cfg_rng.permutation(W_VALUES)
    Type_Theta = np.concatenate([Weights, Reachable])
    Theta[N:Len_Theta] = Type_Theta
    x0 = cfg_rng.beta(2.0, 2.0, size=N)
    episode_configs.append((Theta.copy(), Type_Theta.copy(), x0.copy()))

## Attackers (`BRAttacker` is defined in `Initialize.ipynb`)

In [ ]:
BR_CACHE = {}
def _load_br(def_name):
    if def_name not in BR_CACHE:
        ck = torch.load(os.path.join(SAVE_DIR, "BR_attacker_" + def_name.replace(" ", "_") + ".pth"),
                        map_location=device, weights_only=True)
        att = BRAttacker(rnn_h=ck['config']['RNN_H']).to(device)
        att.load_state_dict(ck['attacker']); att.eval()
        BR_CACHE[def_name] = att
    return BR_CACHE[def_name]


class BRAttackerWrapper:
    def __init__(self, def_name):
        self.att = _load_br(def_name)
    def reset(self, Theta, legal):
        self.h = None
        self.u_prev = np.zeros(N)
        self.theta_t = torch.tensor(Theta, dtype=torch.float32, device=device).view(1, 1, Len_Theta)
        self.legal_mask = torch.zeros(n_u, dtype=torch.bool, device=device)
        self.legal_mask[legal] = True
    def act(self, x):
        obs = np.concatenate([x, self.u_prev]).astype(np.float32)
        with torch.no_grad():
            logits, _, self.h = self.att(torch.tensor(obs, device=device).view(1, 1, 2 * N),
                                         self.theta_t, self.legal_mask.view(1, 1, n_u), self.h)
        u = int(Categorical(logits=logits.view(-1)).sample())
        self.u_prev = actions[u] / B
        return u


def attacker_action(Attacker, a_dist, critic_Q, x_t, m2_t, tt_t, legal):
    # critic_Q() returns the defender's payoff matrix under the critic, computed only when needed
    if Attacker == 'Trained P2':
        with torch.no_grad():
            u_dist = Player2(x_t.unsqueeze(0), m2_t.unsqueeze(0), tt_t.unsqueeze(0)).squeeze(0).cpu().numpy()
        return int(np.random.choice(n_u, p=u_dist / u_dist.sum()))
    Q = critic_Q()
    if Attacker == 'Critic-BR P2':
        return int(legal[np.argmin((a_dist @ Q)[legal])])      # minimizes the defender's expected payoff
    if Attacker == 'Equilibrium P2':
        _, _, q = MatrixValue(Q[:, legal])
        q = np.clip(q, 0, None); q = q / q.sum()
        return int(legal[np.random.choice(len(legal), p=q)])
    raise ValueError(Attacker)

## Rollout

In [ ]:
def run_episodes(Defender, Attacker):
    defw = get_defender(Defender)
    br = BRAttackerWrapper(Defender) if Attacker == 'BR-PPO P2' else None
    ep_disc = []
    for i_episode in tqdm(range(num_episodes), desc=f"{Defender} vs {Attacker}", leave=False):
        np.random.seed(1000 + i_episode); torch.manual_seed(1000 + i_episode); random.seed(1000 + i_episode)
        Theta, Type_Theta, x0 = episode_configs[i_episode]
        legal = legal_actions(Theta[2*N:3*N])
        defw.reset(Theta, Type_Theta, legal)
        if br is not None:
            br.reset(Theta, legal)
        x = x0.copy()
        # The critic's view of the episode, used by the critic-based attackers
        z_s = torch.zeros(Z_S_Size, dtype=torch.float32, device=device)
        Context = np.zeros([max_steps, N * 2], dtype=np.float32)
        m2 = np.full(N * 2, 0.5)
        th_t = torch.tensor(Theta, dtype=torch.float32, device=device)
        tt_t = torch.tensor(Type_Theta, dtype=torch.float32, device=device)
        disc_sum = 0.0
        for t in range(max_steps):
            x_t = torch.tensor(x, dtype=torch.float32, device=device)
            m2_t = torch.tensor(m2, dtype=torch.float32, device=device)

            def critic_Q():
                ctx = torch.tensor(Context, dtype=torch.float32, device=device).view(1, max_steps, N * 2)
                with torch.no_grad():
                    Emb = Context_Model(ctx, Context_Mask(t)).squeeze(0)
                    return critic(z_s.unsqueeze(0), Emb.unsqueeze(0), x_t.unsqueeze(0), m2_t.unsqueeze(0),
                                  tt_t.unsqueeze(0), th_t.unsqueeze(0)).squeeze(0).cpu().numpy().reshape(n_a, n_u)

            a_dist = defw.act_dist(x)
            u = br.act(x) if br is not None else attacker_action(Attacker, a_dist, critic_Q, x_t, m2_t, tt_t, legal)
            a = int(np.random.choice(n_a, p=a_dist))         # the defender samples from its own policy
            a_vec, u_vec = actions[a], actions[u]
            next_x = next_state(x, a_vec, u_vec, Theta)
            r = reward(next_x, Theta)
            disc_sum += (GAMMA ** t) * r
            next_m2 = m2_update(m2, x, u_vec, next_x)
            y = Observation(next_x)
            defw.observe(x, a_vec, u_vec, next_x, y)
            with torch.no_grad():
                z_s = Z_S_ENC(z_s, torch.tensor(y, dtype=torch.float32, device=device),
                              torch.tensor(a_vec, dtype=torch.float32, device=device))
            Context[t, :] = Context_Row(a_vec, y)
            x, m2 = next_x, next_m2
        ep_disc.append(disc_sum)
    return np.asarray(ep_disc)

## Evaluate every defender against every attacker

In [ ]:
ATTACKERS = ['Trained P2', 'Critic-BR P2', 'Equilibrium P2', 'BR-PPO P2']
DEFENDERS = ['Critic P1', 'Trained P1', 'RPPO', 'DR_RPPO', 'VARIBAD', 'PEARL', 'MMD', 'Random P1']

all_runs = {}
for Defender in DEFENDERS:
    all_runs[Defender] = {Attacker: run_episodes(Defender, Attacker) for Attacker in ATTACKERS}
    print(Defender, {k: round(v.mean(), 3) for k, v in all_runs[Defender].items()})

## Worst-case return and empirical exploitability

In [ ]:
worst_runs = {}
for Defender in DEFENDERS:
    runs = all_runs[Defender]
    worst = min(runs, key=lambda k: runs[k].mean())          # attackers minimize the defender's return
    worst_runs[Defender] = (worst, runs[worst])

anchor = max(worst_runs, key=lambda k: worst_runs[k][1].mean())
V_star = worst_runs[anchor][1]

print(f"anchor defender (least exploitable): {anchor}")
print(f"{'Defender':<12}{'worst attacker':>16}{'W':>9}{'Delta':>9}{'SE':>8}")
print("-" * 54)
for Defender in DEFENDERS:
    worst, W = worst_runs[Defender]
    Delta = V_star - W                                         # paired over the shared episodes
    print(f"{Defender:<12}{worst:>16}{W.mean():>9.3f}{Delta.mean():>9.3f}{Delta.std(ddof=1) / np.sqrt(len(Delta)):>8.3f}")